In [6]:
import importlib
importlib.reload(models)

<module 'models' from '/mnt/hd2/clint/ml_results/contrastive_classifiers/SimCLR-2_202529f/models.py'>

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [8]:
target_modules = ["query", "value"]
target_modules.extend([f"mixer.layers.{i}" for i in range(0, 11, 2)])

m = prepare_model_for_kbit_training(m)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.3,
    bias="none",
    task_type="FEATURE_EXTRACTION",  # Changed from SEQ_CLS since we're doing contrastive learning
)

m = get_peft_model(m, lora_config)
state_dict = torch.load(modelf)
m.load_state_dict(state_dict, strict=False)

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
verification_df = pd.read_pickle('280_embedded.pd')

In [ ]:
ablang_hc_hug_path = '/dors/iglab/Members/holtcm/.cache/huggingface/hub/models--qilowoq--AbLang_heavy/snapshots/ecac793b0493f76590ce26d48f7aac4912de8717/'
ablang_lc_hug_path = '/dors/iglab/Members/holtcm/.cache/huggingface/hub/models--qilowoq--AbLang_light/snapshots/ce0637166f5e6e271e906d29a8415d9fdc30e377'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 256
# model_fn = os.path.join(trained_model_folder, best_model_fname)
model_fn = best_model_fname


# Step 1: Load the trained model
# os.chdir(trained_model_folder)
# print(os.getcwd())
import models
import analysis
import data_handling
from get_run_specifics import get_run_specifics

def embed(model, dataloader):
    model.eval()
    all_embeddings = []    
    for batch in dataloader:
        h_seqs, h_mask, l_seqs, l_mask = [b.to(device) for b in batch[:-1]]        
        with torch.no_grad():
            logits, embeddings = model(h_input_ids=h_seqs, h_attention_mask=h_mask, 
                                       l_input_ids=l_seqs, l_attention_mask=l_mask, 
                                       return_embedding=True)
            # Fill the pre-allocated tensors
            all_embeddings.extend(embeddings.cpu().numpy())
            del embeddings, logits, h_seqs, h_mask, l_seqs, l_mask
    return all_embeddings

heavy_tokenizer = AutoTokenizer.from_pretrained(ablang_hc_hug_path)
light_tokenizer = AutoTokenizer.from_pretrained(ablang_lc_hug_path)

# Put together everything I'll need
run_key = "SimCLR2_250129f"
run_specifics = get_run_specifics(run_key)

df = pd.read_pickle("rbd_dataset_16-2_split.pd")



dataloader = data_handling.get_dataloader(heavy_tokenizer, light_tokenizer, df, 16, batch_size=batch_size, shuffle=False)

model = models.setup_model(run_specifics, modelf=best_model_fname).to(device)
all_embeddings = embed(model, dataloader)

with open("all_embeddings.pkl", "wb") as f:
    pickle.dump(all_embeddings, f)

df.loc[:, "EMBEDDING"] = all_embeddings
df.to_pickle("rbd_dataset_16-2_split_embedded.pd")
# model = models.setup_model(run_specifics, modelf=model_fn).to(device)

# train_dataset = torch.